# Interval tally of the Well-Tempered Clavier

This notebook counts how often each harmonic interval (two notes sounding together)
occurs in Bach's *Well-Tempered Clavier*, Books I and II (BWV 846–893).
Each interval is counted separately for every pair of pitch classes, so the major
third C–E and the major third D–F♯ are tallied separately.

The functions used here are in `temperament_functions.py`. See `README.md` for
the background and the assignment.

## 1. Setup

`temperament_functions` provides `process_scores()` (counting) and
`pprint_interval_data()` (printing). It needs the `music21` package.

In [1]:
import json
import os

from temperament_functions import *

## 2. The scores

The folder `scores_xml/` holds 48 MusicXML files, one per prelude–fugue pair,
named by BWV number: `BWV_0846.xml` (Book I, C major) to `BWV_0869.xml`
(Book I, B minor), and `BWV_0870.xml` to `BWV_0893.xml` for Book II.

In [2]:
SCORE_DIR = 'scores_xml/'

## 3. Counting the intervals

`process_scores()` parses every file and adds its intervals to one tally for the
whole corpus. For each score it:

1. *chordifies* the music into vertical slices (a new slice starts whenever any note
   starts or ends);
2. takes every pair of notes in each slice and records it as
   `(lower_pitch_class, upper_pitch_class, interval_class)`, with pitch classes 0–11
   (C = 0) and the interval in semitones modulo 12 (so an octave counts as 0
   and a tenth as a third);
3. adds a weight for each pair:
   - `durationWeight=True`: the length of the slice in quarter notes;
   - `metricWeight=True`: multiplied by the beat strength (1.0 on the downbeat,
     less on weaker beats).

With `total_weight=1.0` the result is scaled so that all values sum to 1: each value is
that interval's share of the whole corpus.

The result is also saved to `wtc_frequencies.json`, so you can load it later
without running the analysis again.

Running this cell takes a few minutes.

In [3]:
wtc_files = sorted([os.path.join(SCORE_DIR, f) for f in os.listdir(SCORE_DIR) if f.endswith('.xml')])
wtc_frequencies = process_scores(
    wtc_files,
    durationWeight=True,
    metricWeight=True,
    total_weight=1.0
)
json_wtc_frequencies = {str(key): value for key, value in wtc_frequencies.items()}
with open('wtc_frequencies.json', 'w') as f:
    json.dump(json_wtc_frequencies, f, indent=2)

## 4. Looking at the results

`pprint_interval_data()` prints the tally three times:

- **Sorted by interval**: grouped by interval class (P1, m2, M2, …), highest share first
  within each group. Use this to compare the 12 transpositions of one interval.
- **Sorted by root note**: grouped by the lower note of the interval.
- **Sorted by weight**: all intervals, highest share first.

How to read a line: `Eb - F# : 0.0176` means that intervals with E♭ as the lower pitch
class and F♯ as the upper one (3 semitones, a minor third) make up 1.76% of the
total weight.

Notes:

- The scores' spelling is not stored, so the names are chosen only for display
  (`number_of_flats=3` gives C C♯ D E♭ E F F♯ G A♭ A B♭ B). The line `Eb - F#` covers
  both D♯–F♯ and E♭–G♭.
- `P1` contains both unisons (two voices on the same note) and octaves (and double
  octaves etc.), because intervals are reduced modulo 12.

In [4]:
pprint_interval_data(wtc_frequencies)

Sorted by interval
--- P1 ---
Ab - Ab : 0.0129
Eb - Eb : 0.0126
Bb - Bb : 0.0121
C# - C# : 0.0118
B  - B  : 0.0105
C  - C  : 0.0105
F  - F  : 0.0097
F# - F# : 0.0097
E  - E  : 0.0083
A  - A  : 0.0074
D  - D  : 0.0069
G  - G  : 0.0055
--- m2 ---
Eb - E  : 0.0017
Ab - A  : 0.0016
F  - F# : 0.0016
Bb - B  : 0.0013
C  - C# : 0.0013
G  - Ab : 0.0012
C# - D  : 0.0009
D  - Eb : 0.0007
B  - C  : 0.0007
E  - F  : 0.0007
A  - Bb : 0.0006
F# - G  : 0.0006
--- M2 ---
C# - Eb : 0.0056
F# - Ab : 0.0055
Ab - Bb : 0.0051
Eb - F  : 0.0049
B  - C# : 0.0045
Bb - C  : 0.0037
E  - F# : 0.0035
A  - B  : 0.0033
F  - G  : 0.0031
C  - D  : 0.0027
G  - A  : 0.0026
D  - E  : 0.0023
--- m3 ---
Eb - F# : 0.0174
Ab - B  : 0.0173
Bb - C# : 0.0159
F  - Ab : 0.0149
C# - E  : 0.0148
C  - Eb : 0.0136
G  - Bb : 0.0132
F# - A  : 0.0129
D  - F  : 0.0117
A  - C  : 0.0103
E  - G  : 0.0101
B  - D  : 0.0096
--- M3 ---
B  - Eb : 0.0140
F# - Bb : 0.0120
C# - F  : 0.0119
E  - Ab : 0.0115
Ab - C  : 0.0112
Bb - D  : 0.0111
A  - C# 